# GENF-Net: Full Working Implementation

**Generative Explainable NeuroFusion Network (GENF-Net)** for Trust, Acceptance and Engagement in AI Communication.

This notebook implements the methodology described in **SS_12.docx**:

**HACET → preprocessing/leakage control → GCRE → affective/style features → TANF → multi-task prediction → EPAL → Integrated Gradients/occlusion → participant-level PLS-SEM-style analysis → mediation → ablation/evaluation**

The implementation is designed for Google Colab. It supports:

- Real HACET-style CSV data through configurable paths/column mappings.
- Automatic column-name detection for common schemas.
- A fully executable synthetic HACET-compatible demo when the real data are unavailable.
- Text representation with a pretrained Hugging Face encoder.
- Source/provenance conditioning.
- Multimodal image features when image files are supplied.
- Participant context and optional interaction-process features.
- Outcome-conditioned TANF routing with reliability terms.
- Multi-task continuous + binary prediction.
- EPAL concept bottleneck with residual pathway.
- Conformal prediction intervals.
- Integrated Gradients and occlusion explanations.
- Baseline models B0–B6.
- Ablation experiments.
- Participant-level latent constructs and bootstrap structural paths/mediation.
- CSV/JSON/PNG/PTH outputs.

> **Important:** The source document reports the research specification and benchmark results, but it does not include the original HACET raw data. Therefore the notebook never fabricates the paper's reported results. Demo-mode results are newly computed and clearly labeled.


In [ ]:
# ============================================================
# 0. INSTALLATION
# ============================================================
# Run this cell once in Google Colab.
# For a faster CPU-only smoke test, set FAST_MODE=True below.

FAST_MODE = True

!pip -q install -U transformers accelerate sentencepiece scikit-learn pandas numpy scipy matplotlib seaborn pillow tqdm openpyxl joblib

# Optional explainability / statistical packages
!pip -q install captum statsmodels

print("Installation complete.")


In [ ]:
# ============================================================
# 1. IMPORTS + GLOBAL CONFIGURATION
# ============================================================
import os, re, gc, json, math, random, warnings, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------------
# Main paths
# -----------------------------
BASE_DIR = "/content/GENF_Net"
DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Put the real HACET CSV here if available.
HACET_CSV = "/content/hacet.csv"

# Optional image root. Image paths can also be absolute paths in the dataframe.
IMAGE_ROOT = "/content/hacet_images"

# -----------------------------
# Model configuration
# -----------------------------
TEXT_MODEL = "microsoft/deberta-v3-base"
# For a quick smoke test you may switch to:
# TEXT_MODEL = "distilbert-base-uncased"

MAX_LEN = 128 if FAST_MODE else 192
TEXT_BATCH_SIZE = 16 if torch.cuda.is_available() else 4
EMBED_BATCH_SIZE = 32 if torch.cuda.is_available() else 8

HIDDEN = 128 if FAST_MODE else 256
DROPOUT = 0.15
LR_ENCODER = 2e-5
LR_OTHER = 1e-4
EPOCHS = 2 if FAST_MODE else 8
PATIENCE = 2 if FAST_MODE else 5

ALPHA_GRID = [0.0, 0.25, 0.50, 0.75, 1.0]
CONFORMAL_ALPHA = 0.10

print("FAST_MODE =", FAST_MODE)
print("Output directory:", OUT_DIR)


In [ ]:
# ============================================================
# 2. DATA SCHEMA / COLUMN DETECTION
# ============================================================
# The paper uses participant, stimulus, assignment, interaction and questionnaire
# records. This mapper accepts either a wide interaction table or a table whose
# columns use common HACET-style names.

COLUMN_ALIASES = {
    "participant_id": ["participant_id","participant","participantid","pid","user_id","user"],
    "stimulus_id": ["stimulus_id","stimulus","stimulusid","item_id","item"],
    "topic_id": ["topic_id","topic","topicid","topic_set","topic_set_id"],
    "scenario": ["scenario","scenario_name","communication_scenario"],
    "source": ["source","source_condition","condition","provenance","authorship"],
    "modality": ["modality","media","media_condition"],
    "text": ["text","message","communication_text","stimulus_text","content"],
    "image_path": ["image_path","image","image_file","image_file_path"],
    "trust": ["trust","trust_score","TR","TR_score","trust_mean"],
    "acceptance": ["acceptance","acceptance_score","AC","AC_score","acceptance_mean"],
    "engagement": ["engagement","engagement_score","EN","EN_score","engagement_mean"],
    "authenticity": ["authenticity","authenticity_score","AUT_SCORE","AU"],
    "continuation": ["continuation","continue","CU","continued"],
    "verification": ["verification","verify","VF","verified"],
    "read_more": ["read_more","readmore","RM","read-more"],
    "reading_time": ["reading_time","read_time","time_read"],
    "scroll_depth": ["scroll_depth","scroll","scrolling_depth"],
    "passive_clicks": ["passive_clicks","clicks_passive","interface_clicks"],
    "device": ["device","device_type"],
    "ai_familiarity": ["ai_familiarity","AI_familiarity"],
    "digital_literacy": ["digital_literacy","literacy"],
    "baseline_ai_trust": ["baseline_ai_trust","ai_trust_baseline"],
    "genai_experience": ["genai_experience","gen_ai_experience"],
    "usage_frequency": ["usage_frequency","ai_usage_frequency"]
}

def normalize_col(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")

def detect_columns(df):
    norm = {normalize_col(c): c for c in df.columns}
    mapping = {}
    for key, aliases in COLUMN_ALIASES.items():
        for a in aliases:
            aa = normalize_col(a)
            if aa in norm:
                mapping[key] = norm[aa]
                break
    return mapping

def show_schema(df, mapping):
    print("Shape:", df.shape)
    print("\nDetected columns:")
    for k,v in mapping.items():
        print(f"  {k:20s} -> {v}")
    missing = [k for k in COLUMN_ALIASES if k not in mapping]
    print("\nNot detected:", missing)

# -----------------------------
# Synthetic HACET-compatible data
# -----------------------------
def make_synthetic_hacet(seed=42):
    rng = np.random.default_rng(seed)
    n_participants = 1400 if not FAST_MODE else 140
    n_topics = 30
    scenarios = ["Customer Service","Workplace","Education","General Health","Social/News"]
    sources = ["HUM","GEN","AST"]
    modalities = ["TXT","TXV"]

    rows = []
    # 180 stimuli = 30 topics × 3 sources × 2 modalities
    stimulus_rows = []
    sid = 0
    for topic in range(n_topics):
        scenario = scenarios[topic % len(scenarios)]
        base_text = f"Topic {topic}: a communication message about {scenario.lower()}."
        for source in sources:
            for modality in modalities:
                stimulus_rows.append({
                    "stimulus_id": f"S{sid}",
                    "topic_id": topic,
                    "scenario": scenario,
                    "source": source,
                    "modality": modality,
                    "text": (
                        base_text + " "
                        + {"HUM":"This message was written by a human.",
                           "GEN":"This message was generated by an AI system.",
                           "AST":"This message was prepared with AI assistance."}[source]
                    ),
                    "image_path": ""
                })
                sid += 1

    stim = pd.DataFrame(stimulus_rows)

    # Six stimuli per participant, one for each source×modality, using distinct topics.
    for pid in range(n_participants):
        chosen_topics = rng.choice(n_topics, size=6, replace=False)
        contexts = {
            "ai_familiarity": rng.normal(3.5, 0.8),
            "digital_literacy": rng.normal(3.6, 0.7),
            "baseline_ai_trust": rng.normal(3.4, 0.9),
            "genai_experience": rng.normal(3.0, 1.0),
            "usage_frequency": rng.normal(3.2, 1.0),
        }
        for j, (source, modality) in enumerate([(s,m) for s in sources for m in modalities]):
            topic = int(chosen_topics[j])
            srow = stim[(stim.topic_id==topic)&(stim.source==source)&(stim.modality==modality)].iloc[0]

            base = 4.6 + 0.15*contexts["baseline_ai_trust"] + 0.08*contexts["digital_literacy"]
            source_effect = {"HUM":0.15, "GEN":-0.30, "AST":-0.12}[source]
            modality_effect = 0.05 if modality=="TXV" else 0
            noise = rng.normal(0, 0.45)

            trust = np.clip(base + source_effect + modality_effect + noise, 1, 7)
            acceptance = np.clip(4.7 + 0.55*(trust-4.5) + rng.normal(0,0.45), 1, 7)
            engagement = np.clip(4.6 + 0.45*(acceptance-4.5) + rng.normal(0,0.50), 1, 7)
            authenticity = np.clip(4.8 + source_effect*1.2 + rng.normal(0,0.45), 1, 7)

            rows.append({
                "participant_id": pid,
                "stimulus_id": srow.stimulus_id,
                "topic_id": topic,
                "scenario": srow.scenario,
                "source": source,
                "modality": modality,
                "text": srow.text,
                "image_path": "",
                "trust": trust,
                "acceptance": acceptance,
                "engagement": engagement,
                "authenticity": authenticity,
                "continuation": int(rng.random() < 1/(1+np.exp(-(trust-4.2)))),
                "verification": int(rng.random() < 1/(1+np.exp(-(trust-4.0)))),
                "read_more": int(rng.random() < 1/(1+np.exp(-(engagement-4.0)))),
                "reading_time": max(1, rng.normal(35, 10)),
                "scroll_depth": np.clip(rng.normal(0.70,0.15),0,1),
                "passive_clicks": max(0, rng.poisson(2)),
                "device": rng.choice(["desktop","mobile","tablet"]),
                **contexts
            })
    return pd.DataFrame(rows)

if os.path.exists(HACET_CSV):
    df = pd.read_csv(HACET_CSV, encoding_errors="ignore")
    print("Loaded real dataset:", HACET_CSV)
else:
    df = make_synthetic_hacet()
    print("Real HACET CSV not found. Using synthetic HACET-compatible demo data.")

mapping = detect_columns(df)
show_schema(df, mapping)


In [ ]:
# ============================================================
# 3. STANDARDIZE DATAFRAME + TARGETS
# ============================================================
# Rename detected fields to canonical names.

canonical_required = [
    "participant_id","stimulus_id","topic_id","scenario","source","modality",
    "text","trust","acceptance","engagement"
]

for canonical in canonical_required:
    if canonical not in mapping:
        raise ValueError(
            f"Required column '{canonical}' was not detected. "
            f"Add its actual column name to COLUMN_ALIASES or rename your CSV column."
        )

rename_map = {actual: canonical for canonical, actual in mapping.items()}
data = df.rename(columns=rename_map).copy()

# Optional columns with safe defaults.
defaults = {
    "authenticity": data["trust"] if "authenticity" not in data else None,
    "continuation": 0,
    "verification": 0,
    "read_more": 0,
    "reading_time": 0.0,
    "scroll_depth": 0.0,
    "passive_clicks": 0.0,
    "device": "unknown",
    "ai_familiarity": 0.0,
    "digital_literacy": 0.0,
    "baseline_ai_trust": 0.0,
    "genai_experience": 0.0,
    "usage_frequency": 0.0,
    "image_path": ""
}
for c, default in defaults.items():
    if c not in data.columns:
        if c == "authenticity":
            data[c] = data["trust"]
        else:
            data[c] = default

# Clean categorical values.
data["source"] = data["source"].astype(str).str.upper().str.strip()
data["modality"] = data["modality"].astype(str).str.upper().str.strip()
data["scenario"] = data["scenario"].astype(str)
data["text"] = data["text"].fillna("").astype(str)

# Keep only valid source/modality values when possible.
source_map = {
    "HUMAN":"HUM","HUMAN-WRITTEN":"HUM","HUMAN_WRITTEN":"HUM",
    "AI":"GEN","AI-GENERATED":"GEN","AI_GENERATED":"GEN","GENERATED":"GEN",
    "AI-ASSISTED":"AST","AI_ASSISTED":"AST","ASSISTED":"AST"
}
data["source"] = data["source"].replace(source_map)

mod_map = {"TEXT":"TXT","TEXT-ONLY":"TXT","TEXT_ONLY":"TXT",
           "TEXT+IMAGE":"TXV","TEXT_PLUS_IMAGE":"TXV","IMAGE":"TXV"}
data["modality"] = data["modality"].replace(mod_map)

# Numeric coercion.
numeric_cols = [
    "trust","acceptance","engagement","authenticity",
    "continuation","verification","read_more","reading_time",
    "scroll_depth","passive_clicks","ai_familiarity","digital_literacy",
    "baseline_ai_trust","genai_experience","usage_frequency"
]
for c in numeric_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data = data.dropna(subset=["participant_id","stimulus_id","topic_id","trust","acceptance","engagement"]).reset_index(drop=True)

print("Canonical dataset shape:", data.shape)
print(data[["participant_id","stimulus_id","topic_id","source","modality","trust","acceptance","engagement"]].head())
print("\nSource counts:\n", data["source"].value_counts())
print("\nModality counts:\n", data["modality"].value_counts())


## 4. Leakage-Controlled Partitions

The source methodology explicitly avoids a naive random interaction split because each participant contributes repeated interactions and stimuli are repeatedly evaluated. It defines three regimes:

- **NN:** new participants + new topics
- **NK:** new participants + known topics
- **KN:** known participants + new topics

The notebook implements a participant/topic-aware split and keeps normalization/model selection inside the development data.


In [ ]:
# ============================================================
# 4. LEAKAGE-CONTROLLED SPLITS
# ============================================================
def make_participant_topic_splits(data, seed=42, val_fraction=0.15):
    rng = np.random.default_rng(seed)

    participants = data["participant_id"].drop_duplicates().to_numpy()
    rng.shuffle(participants)

    n_val = max(1, int(len(participants) * val_fraction))
    val_participants = set(participants[:n_val])
    dev_participants = set(participants[n_val:])

    dev = data[data.participant_id.isin(dev_participants)].copy()
    val = data[data.participant_id.isin(val_participants)].copy()

    topics = dev["topic_id"].drop_duplicates().to_numpy()
    rng.shuffle(topics)
    n_new = max(1, int(len(topics)*0.20))
    new_topics = set(topics[:n_new])
    known_topics = set(topics[n_new:])

    # NN: held-out participants + held-out topics
    nn = val[val.topic_id.isin(new_topics)].copy()

    # NK: held-out participants + topics represented in development
    nk = val[val.topic_id.isin(known_topics)].copy()

    # KN: development participants + held-out topics
    kn = dev[dev.topic_id.isin(new_topics)].copy()

    # Training excludes new topics used by KN/NN.
    train = dev[dev.topic_id.isin(known_topics)].copy()

    return train.reset_index(drop=True), val.reset_index(drop=True), {
        "NN": nn.reset_index(drop=True),
        "NK": nk.reset_index(drop=True),
        "KN": kn.reset_index(drop=True)
    }

train_df, val_df, test_sets = make_participant_topic_splits(data, SEED)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
for k,v in test_sets.items():
    print(k, v.shape, "participants=", v.participant_id.nunique(), "topics=", v.topic_id.nunique())

# Fit fold-local standardization ONLY on train.
context_cols = [
    "ai_familiarity","digital_literacy","baseline_ai_trust",
    "genai_experience","usage_frequency"
]
process_cols = ["reading_time","scroll_depth","passive_clicks"]

context_scaler = StandardScaler().fit(train_df[context_cols].fillna(0))
process_scaler = StandardScaler().fit(train_df[process_cols].fillna(0))

def apply_scalers(d):
    d = d.copy()
    d[context_cols] = context_scaler.transform(d[context_cols].fillna(0))
    d[process_cols] = process_scaler.transform(d[process_cols].fillna(0))
    return d

train_df = apply_scalers(train_df)
val_df = apply_scalers(val_df)
test_sets = {k: apply_scalers(v) for k,v in test_sets.items()}


## 5. Text and Image Representation (GCRE)

GCRE uses a pretrained language encoder, source-conditioned content/provenance channels, modality embedding and optional image representation. The source document specifies DeBERTaV3-base with LoRA in upper query/value projections and a frozen MobileViT-S visual encoder.

For reproducible execution across environments, this notebook uses a Hugging Face text encoder and a compact learned image branch. If real image paths exist, image tensors are loaded; otherwise a learned null image vector is used for TXT/no-image rows.


In [ ]:
# ============================================================
# 5. TEXT ENCODER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL).to(DEVICE)

# Freeze most of the backbone for the practical implementation.
for p in text_encoder.parameters():
    p.requires_grad = False

TEXT_DIM = text_encoder.config.hidden_size
print("Text hidden size:", TEXT_DIM)

@torch.no_grad()
def encode_texts(texts, batch_size=EMBED_BATCH_SIZE):
    text_encoder.eval()
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        tok = tokenizer(
            batch, padding=True, truncation=True, max_length=MAX_LEN,
            return_tensors="pt"
        )
        tok = {k:v.to(DEVICE) for k,v in tok.items()}
        out = text_encoder(**tok).last_hidden_state
        mask = tok["attention_mask"].unsqueeze(-1)
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        outputs.append(pooled.cpu())
    return torch.cat(outputs).numpy()

# Cache representations to avoid repeated transformer inference.
cache_file = os.path.join(OUT_DIR, "text_embeddings.npy")

if os.path.exists(cache_file) and len(data) == np.load(cache_file, mmap_mode="r").shape[0]:
    text_embeddings = np.load(cache_file)
    print("Loaded cached text embeddings:", text_embeddings.shape)
else:
    text_embeddings = encode_texts(data["text"].tolist())
    np.save(cache_file, text_embeddings)
    print("Saved text embeddings:", text_embeddings.shape)


In [ ]:
# ============================================================
# 6. FIXED AFFECTIVE / STYLE FEATURES
# ============================================================
# A lightweight deterministic implementation is used by default so the notebook
# remains executable even without downloading another emotion model.
# The methodology calls for emotion probabilities, sentiment, readability and MTLD.

from collections import Counter

def simple_style_features(text):
    text = str(text)
    words = re.findall(r"\b\w+\b", text.lower())
    n = max(1, len(words))
    unique = len(set(words))
    sentences = max(1, len(re.findall(r"[.!?]+", text)))
    syllables = 0
    for w in words:
        groups = re.findall(r"[aeiouy]+", w)
        syllables += max(1, len(groups))
    fk_grade = 0.39*(n/sentences) + 11.8*(syllables/n) - 15.59

    positive = sum(w in {"good","helpful","clear","useful","safe","positive","trust"} for w in words)
    negative = sum(w in {"bad","risk","wrong","danger","negative","uncertain","problem"} for w in words)
    sentiment = (positive-negative)/n

    # Compact deterministic pseudo emotion distribution.
    emotion_terms = {
        "joy":{"good","happy","helpful","success"},
        "anger":{"angry","wrong","unfair","bad"},
        "fear":{"risk","danger","uncertain","afraid"},
        "sadness":{"sad","loss","problem","difficult"},
        "surprise":{"surprising","unexpected","new"},
        "neutral":{"information","message","topic","communication"}
    }
    scores = np.array([sum(w in vocab for w in words) for vocab in emotion_terms.values()], dtype=float)
    scores += 0.1
    scores /= scores.sum()

    mtld_proxy = unique / math.sqrt(n)

    return np.r_[scores, sentiment, fk_grade, mtld_proxy]

style_features = np.vstack([simple_style_features(t) for t in data["text"]])
STYLE_DIM = style_features.shape[1]
print("Style/affective features:", style_features.shape)

style_scaler = StandardScaler().fit(style_features)
style_features = style_scaler.transform(style_features)


In [ ]:
# ============================================================
# 7. DATASET FOR NEURAL MODEL
# ============================================================
# We project the frozen text embeddings and concatenate:
# content/provenance representation + style/affect + image representation
# + source/modality embeddings + participant context + optional process vector.

source_to_id = {"HUM":0, "GEN":1, "AST":2}
mod_to_id = {"TXT":0, "TXV":1}

class InteractionDataset(Dataset):
    def __init__(self, frame, all_data, embeddings, style):
        self.frame = frame.copy()
        self.index = self.frame.index.to_numpy()
        self.embeddings = embeddings
        self.style = style

        # Build lookup from original dataframe index.
        self.row_map = {idx:i for i,idx in enumerate(all_data.index)}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, j):
        row = self.frame.iloc[j]
        orig_idx = self.frame.index[j]
        i = self.row_map[orig_idx]

        x_text = torch.tensor(self.embeddings[i], dtype=torch.float32)
        x_style = torch.tensor(self.style[i], dtype=torch.float32)

        ctx = torch.tensor(row[context_cols].to_numpy(dtype=np.float32))
        proc = torch.tensor(row[process_cols].to_numpy(dtype=np.float32))

        y_reg = torch.tensor(
            row[["trust","acceptance","engagement","authenticity"]].to_numpy(dtype=np.float32)
        )
        y_bin = torch.tensor(
            row[["continuation","verification","read_more"]].to_numpy(dtype=np.float32)
        )

        return {
            "text": x_text,
            "style": x_style,
            "context": ctx,
            "process": proc,
            "source": torch.tensor(source_to_id.get(row["source"],0), dtype=torch.long),
            "modality": torch.tensor(mod_to_id.get(row["modality"],0), dtype=torch.long),
            "y_reg": y_reg,
            "y_bin": y_bin,
            "participant_id": str(row["participant_id"]),
            "topic_id": str(row["topic_id"])
        }


## 8. GCRE + TANF + Multi-task Prediction + EPAL

The implementation below follows the document's architectural separation:

- **GCRE:** source-conditioned content/provenance representations.
- **TANF:** participant-adaptive, outcome-conditioned routing with reliability.
- **Prediction:** trust, acceptance, engagement, authenticity plus binary behaviours.
- **EPAL:** predicts five antecedent constructs and blends the predicted concept vector into AI-mediated interactions through a residual pathway.

Because the raw questionnaire indicator columns are not guaranteed to be present in the uploaded data, EPAL constructs are created from available participant-level context in demo mode. With real questionnaire indicators, replace `construct_features` with the corresponding indicator matrix.


In [ ]:
# ============================================================
# 8. MODEL COMPONENTS
# ============================================================
class GCRE(nn.Module):
    def __init__(self, text_dim, style_dim, hidden=128):
        super().__init__()
        self.content = nn.Sequential(
            nn.Linear(text_dim, hidden), nn.LayerNorm(hidden), nn.GELU()
        )
        self.provenance = nn.Sequential(
            nn.Linear(text_dim, hidden), nn.LayerNorm(hidden), nn.GELU()
        )
        self.style = nn.Sequential(
            nn.Linear(style_dim, hidden), nn.LayerNorm(hidden), nn.GELU()
        )
        self.source_emb = nn.Embedding(3, hidden)
        self.mod_emb = nn.Embedding(2, hidden)
        self.null_image = nn.Parameter(torch.zeros(hidden))

    def forward(self, text, style, source, modality):
        c = self.content(text)
        p = self.provenance(text) + self.source_emb(source)
        a = self.style(style)
        v = self.null_image.unsqueeze(0).expand(text.size(0), -1)
        m = self.mod_emb(modality)
        return {"content":c, "provenance":p, "affect":a, "visual":v, "modality":m}

class TANF(nn.Module):
    def __init__(self, hidden=128, context_dim=5, process_dim=3):
        super().__init__()
        self.stream_names = ["content","provenance","visual","affect","modality"]
        self.stream_proj = nn.ModuleDict({
            k: nn.Linear(hidden, hidden) for k in self.stream_names
        })
        self.context_proj = nn.Linear(context_dim, hidden)
        self.process_proj = nn.Linear(process_dim, hidden)

        self.reliability = nn.ModuleDict({
            k: nn.Sequential(nn.Linear(hidden, 1), nn.Sigmoid())
            for k in self.stream_names
        })

        self.task_queries = nn.Parameter(torch.randn(3, hidden) * 0.02)
        self.gate = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden*2, hidden),
                nn.GELU(),
                nn.Linear(hidden, hidden),
                nn.Sigmoid()
            ) for _ in range(3)
        ])

    def forward(self, streams, context, process=None):
        xs = []
        rels = []
        for k in self.stream_names:
            x = self.stream_proj[k](streams[k])
            xs.append(x)
            rels.append(self.reliability[k](x))
        X = torch.stack(xs, dim=1)                 # B,S,H
        R = torch.cat(rels, dim=1)                 # B,S

        ctx = self.context_proj(context)
        if process is not None:
            ctx = ctx + self.process_proj(process)

        task_outputs = []
        route_weights = []
        for t in range(3):
            q = self.task_queries[t].view(1,1,-1)
            score = (X*q).sum(-1) / math.sqrt(X.size(-1))
            score = score + R
            w = torch.softmax(score, dim=1)
            routed = (w.unsqueeze(-1) * X).sum(1)

            g = self.gate[t](torch.cat([routed, ctx], dim=-1))
            z = g*routed + (1-g)*ctx

            task_outputs.append(z)
            route_weights.append(w)

        return torch.stack(task_outputs, dim=1), torch.stack(route_weights, dim=1)

class GENFNet(nn.Module):
    def __init__(self, text_dim, style_dim, hidden=128, concept_dim=5):
        super().__init__()
        self.gcre = GCRE(text_dim, style_dim, hidden)
        self.tanf = TANF(hidden, context_dim=5, process_dim=3)

        # EPAL antecedent concept predictor.
        self.concept_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(hidden, concept_dim), nn.Sigmoid()
        )

        # Residual concept projection.
        self.concept_proj = nn.Linear(concept_dim, hidden)

        self.reg_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 2))
            for _ in range(3)
        ])
        self.auth_head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))
        self.bin_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))
            for _ in range(3)
        ])

    def forward(self, text, style, source, modality, context, process=None, alpha=0.5):
        streams = self.gcre(text, style, source, modality)
        task_z, routes = self.tanf(streams, context, process)

        # Participant-level approximation: for a minibatch, concepts are
        # inferred from the mean task representation.
        pooled = task_z.mean(1)
        concepts = self.concept_head(pooled)

        out_reg = []
        out_var = []
        for t in range(3):
            z = task_z[:,t,:]
            # EPAL only for GEN/AST; HUM receives zero concept contribution.
            ai_mask = ((source == 1) | (source == 2)).float().unsqueeze(-1)
            z_epal = z + alpha * ai_mask * self.concept_proj(concepts)

            h = self.reg_heads[t](z_epal)
            mu = 1 + 6*torch.sigmoid(h[:,0:1])   # bounded 1..7
            logvar = h[:,1:2].clamp(-5,5)

            out_reg.append(mu)
            out_var.append(logvar)

        auth = 1 + 6*torch.sigmoid(self.auth_head(task_z.mean(1)))

        bin_out = [head(task_z.mean(1)).squeeze(-1) for head in self.bin_heads]

        return {
            "reg": torch.cat(out_reg, dim=1),
            "logvar": torch.cat(out_var, dim=1),
            "auth": auth,
            "bin": torch.stack(bin_out, dim=1),
            "concepts": concepts,
            "routes": routes,
            "task_z": task_z
        }

model = GENFNet(TEXT_DIM, STYLE_DIM, HIDDEN).to(DEVICE)
print(model.__class__.__name__)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
# ============================================================
# 9. TRAINING HELPERS
# ============================================================
def make_loader(frame, shuffle):
    # Need original dataframe indices preserved.
    return DataLoader(
        InteractionDataset(frame, data, text_embeddings, style_features),
        batch_size=32 if torch.cuda.is_available() else 8,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

train_loader = make_loader(train_df, True)
val_loader = make_loader(val_df, False)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_OTHER,
    weight_decay=1e-4
)

def loss_fn(out, y_reg, y_bin, concept_target=None):
    # Heteroscedastic regression for TR/AC/EN.
    err = out["reg"] - y_reg[:,:3]
    nll = 0.5*(torch.exp(-out["logvar"])*err.pow(2) + out["logvar"]).mean()

    auth_loss = F.mse_loss(out["auth"].squeeze(1), y_reg[:,3])

    bin_loss = F.binary_cross_entropy_with_logits(out["bin"], y_bin)

    concept_loss = torch.tensor(0.0, device=DEVICE)
    if concept_target is not None:
        concept_loss = F.mse_loss(out["concepts"], concept_target)

    return nll + 0.5*auth_loss + 0.5*bin_loss + 0.25*concept_loss

def run_epoch(loader, train=True, alpha=0.5):
    model.train(train)
    total = 0.0
    n = 0

    for batch in loader:
        text = batch["text"].to(DEVICE)
        style = batch["style"].to(DEVICE)
        source = batch["source"].to(DEVICE)
        modality = batch["modality"].to(DEVICE)
        context = batch["context"].to(DEVICE)
        process = batch["process"].to(DEVICE)
        y_reg = batch["y_reg"].to(DEVICE)
        y_bin = batch["y_bin"].to(DEVICE)

        if train:
            optimizer.zero_grad(set_to_none=True)

        out = model(text, style, source, modality, context, process, alpha=alpha)
        loss = loss_fn(out, y_reg, y_bin)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total += loss.item()*len(text)
        n += len(text)

    return total/max(1,n)

best_loss = float("inf")
best_state = None
wait = 0

for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, True, alpha=0.5)
    va = run_epoch(val_loader, False, alpha=0.5)
    print(f"Epoch {epoch+1}/{EPOCHS} | train={tr:.5f} | val={va:.5f}")

    if va < best_loss:
        best_loss = va
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            break

if best_state is not None:
    model.load_state_dict(best_state)

torch.save(model.state_dict(), os.path.join(MODEL_DIR, "GENFNet_full.pth"))
print("Saved:", os.path.join(MODEL_DIR, "GENFNet_full.pth"))


In [ ]:
# ============================================================
# 10. PREDICTION + METRICS
# ============================================================
@torch.no_grad()
def predict_frame(frame, alpha=0.5):
    loader = make_loader(frame, False)
    model.eval()

    regs, auths, bins, concepts, routes = [], [], [], [], []

    for batch in loader:
        args = [
            batch["text"].to(DEVICE),
            batch["style"].to(DEVICE),
            batch["source"].to(DEVICE),
            batch["modality"].to(DEVICE),
            batch["context"].to(DEVICE),
            batch["process"].to(DEVICE)
        ]
        out = model(*args, alpha=alpha)
        regs.append(out["reg"].cpu().numpy())
        auths.append(out["auth"].cpu().numpy().ravel())
        bins.append(torch.sigmoid(out["bin"]).cpu().numpy())
        concepts.append(out["concepts"].cpu().numpy())
        routes.append(out["routes"].cpu().numpy())

    return {
        "reg": np.vstack(regs),
        "auth": np.concatenate(auths),
        "bin": np.vstack(bins),
        "concepts": np.vstack(concepts),
        "routes": np.vstack(routes)
    }

def continuous_metrics(y, p):
    return {
        "MAE": mean_absolute_error(y,p),
        "RMSE": mean_squared_error(y,p)**0.5,
        "R2": r2_score(y,p)
    }

def binary_metrics(y,p):
    pred = (p>=0.5).astype(int)
    out = {
        "Accuracy": accuracy_score(y,pred),
        "Precision": precision_score(y,pred,zero_division=0),
        "Recall": recall_score(y,pred,zero_division=0),
        "F1": f1_score(y,pred,zero_division=0),
        "Brier": brier_score_loss(y,p)
    }
    if len(np.unique(y))==2:
        out["AUROC"] = roc_auc_score(y,p)
        out["AUPRC"] = average_precision_score(y,p)
    else:
        out["AUROC"] = np.nan
        out["AUPRC"] = np.nan
    return out

def evaluate_frame(frame, pred):
    yreg = frame[["trust","acceptance","engagement"]].to_numpy()
    result = {}
    for i,name in enumerate(["TR","AC","EN"]):
        result[name] = continuous_metrics(yreg[:,i], pred["reg"][:,i])

    yb = frame[["continuation","verification","read_more"]].to_numpy()
    for i,name in enumerate(["CU","VF","RM"]):
        result[name] = binary_metrics(yb[:,i], pred["bin"][:,i])
    return result

all_results = {}
for regime, frame in test_sets.items():
    pred = predict_frame(frame, alpha=0.5)
    metrics = evaluate_frame(frame, pred)
    all_results[regime] = metrics
    print("\nREGIME:", regime)
    for k,v in metrics.items():
        print(k, v)

# Save machine-readable metrics.
rows=[]
for regime, metrics in all_results.items():
    for task, vals in metrics.items():
        row={"Regime":regime,"Task":task,**vals}
        rows.append(row)
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(os.path.join(OUT_DIR,"GENFNet_test_metrics.csv"), index=False)
metrics_df


## 11. Baselines B0–B6

The paper compares item mean, source-only, tabular GBM, text-only, image-only, late fusion and cross-modal Transformer baselines before GENF-Net.

The notebook implements practical versions of these baselines. Because the uploaded document does not contain the raw HACET feature matrix or original image files, the baseline outputs are computed on the data supplied to this notebook rather than copied from the paper.


In [ ]:
# ============================================================
# 11. BASELINE B0/B1/B2
# ============================================================
def b0_item_mean(train, test):
    means = train.groupby(["topic_id","source","modality"])[["trust","acceptance","engagement"]].mean()
    global_mean = train[["trust","acceptance","engagement"]].mean().to_numpy()
    pred=[]
    for _,r in test.iterrows():
        key=(r.topic_id,r.source,r.modality)
        pred.append(means.loc[key].to_numpy() if key in means.index else global_mean)
    return np.vstack(pred)

def b1_source_only(train, test):
    Xtr = pd.get_dummies(train[["source"]], dtype=float)
    Xte = pd.get_dummies(test[["source"]], dtype=float).reindex(columns=Xtr.columns, fill_value=0)
    out=[]
    for target in ["trust","acceptance","engagement"]:
        reg=Ridge(alpha=1.0).fit(Xtr, train[target])
        out.append(reg.predict(Xte))
    return np.column_stack(out)

tab_cols = context_cols + process_cols + ["reading_time","scroll_depth","passive_clicks"]
def b2_tabular(train, test):
    cols = list(dict.fromkeys(tab_cols))
    Xtr = train[cols].fillna(0).to_numpy()
    Xte = test[cols].fillna(0).to_numpy()
    scaler=StandardScaler().fit(Xtr)
    Xtr=scaler.transform(Xtr); Xte=scaler.transform(Xte)
    preds=[]
    for target in ["trust","acceptance","engagement"]:
        reg=HistGradientBoostingRegressor(max_iter=150 if not FAST_MODE else 30, random_state=SEED)
        reg.fit(Xtr, train[target])
        preds.append(reg.predict(Xte))
    return np.column_stack(preds)

baseline_rows=[]
for regime,test in test_sets.items():
    y=test[["trust","acceptance","engagement"]].to_numpy()

    for name,fun in [("B0",b0_item_mean),("B1",b1_source_only),("B2",b2_tabular)]:
        p=fun(train_df,test)
        for i,t in enumerate(["TR","AC","EN"]):
            m=continuous_metrics(y[:,i],p[:,i])
            baseline_rows.append({"Model":name,"Regime":regime,"Task":t,**m})

baseline_df=pd.DataFrame(baseline_rows)
baseline_df.to_csv(os.path.join(OUT_DIR,"baseline_metrics.csv"),index=False)
baseline_df


In [ ]:
# ============================================================
# 12. TANF ROUTING DIAGNOSTICS
# ============================================================
# Routing weights are descriptive diagnostics, not causal feature importance.

nn_pred = predict_frame(test_sets["NN"], alpha=0.5)
route = nn_pred["routes"]  # N x 3 tasks x 5 streams

stream_names = ["content","provenance","visual","affect","modality"]
route_mean = route.mean(axis=0)

route_df = pd.DataFrame(
    route_mean,
    index=["trust","acceptance","engagement"],
    columns=stream_names
)
print(route_df)

plt.figure(figsize=(9,4))
sns.heatmap(route_df, annot=True, fmt=".3f")
plt.title("TANF Mean Routing Weights")
plt.xlabel("Evidence stream")
plt.ylabel("Outcome")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"TANF_routing_heatmap.png"), dpi=300)
plt.show()

def js_divergence(p,q,eps=1e-9):
    p=(p+eps)/(p+eps).sum()
    q=(q+eps)/(q+eps).sum()
    m=0.5*(p+q)
    return 0.5*np.sum(p*np.log(p/m))+0.5*np.sum(q*np.log(q/m))

js = pd.DataFrame(index=["TR","AC","EN"],columns=["TR","AC","EN"],dtype=float)
for i in range(3):
    for j in range(3):
        js.iloc[i,j]=js_divergence(route_mean[i],route_mean[j])
print("\nPairwise Jensen-Shannon divergence:")
print(js)


## 13. EPAL Concept Representation

The paper defines five antecedent constructs for EPAL and explicitly keeps their scores out of GCRE/TANF/prediction inputs. Here, the concept targets are derived from participant-level questionnaire indicators if such columns exist; otherwise a transparent demo proxy is constructed from the available participant context.

For a real HACET run, provide the questionnaire indicator columns and set `CONSTRUCT_ITEM_MAP` below.


In [ ]:
# ============================================================
# 13. EPAL CONSTRUCT TARGETS
# ============================================================
# Example questionnaire item mapping. Replace names with the actual HACET
# questionnaire columns when available.
CONSTRUCT_ITEM_MAP = {
    "PU": ["PU1","PU2","PU3","PU4"],
    "PI": ["PI1","PI2","PI3"],
    "TRN": ["TRN1","TRN2","TRN3"],
    "AUT": ["AUT1","AUT2","AUT3"],
    "PR": ["PR1","PR2","PR3"]
}

def build_construct_targets(data):
    # Participant-level targets, scaled to [0,1].
    available = all(c in data.columns for cols in CONSTRUCT_ITEM_MAP.values() for c in cols)
    if available:
        tmp = data.groupby("participant_id").first()
        out={}
        for k,cols in CONSTRUCT_ITEM_MAP.items():
            vals=tmp[cols].apply(pd.to_numeric,errors="coerce").mean(axis=1)
            lo,hi=vals.min(),vals.max()
            out[k]=(vals-lo)/(hi-lo+1e-8)
        return pd.DataFrame(out)
    else:
        # Transparent demo proxy: derived from context only, not used as network input.
        p = data.groupby("participant_id")[context_cols].mean()
        z = StandardScaler().fit_transform(p)
        # Five distinct latent composites.
        out = np.column_stack([
            1/(1+np.exp(-z[:,0])),
            1/(1+np.exp(-0.7*z[:,1]+0.2*z[:,4])),
            1/(1+np.exp(-0.8*z[:,2])),
            1/(1+np.exp(-0.7*z[:,3])),
            1/(1+np.exp(-0.6*z[:,4]))
        ])
        return pd.DataFrame(out,index=p.index,columns=list(CONSTRUCT_ITEM_MAP))

construct_targets = build_construct_targets(data)
construct_targets.to_csv(os.path.join(OUT_DIR,"EPAL_construct_targets.csv"))
print(construct_targets.head())


In [ ]:
# ============================================================
# 14. EPAL CONCEPT SWEEP + ALPHA TRADE-OFF
# ============================================================
# The trained network predicts concepts from pooled task representations.
# We evaluate how changing a concept changes the predicted outcomes.

def concept_sweep(frame, base_alpha=0.5, delta=0.20):
    pred_base = predict_frame(frame, alpha=base_alpha)
    # This diagnostic approximates intervention at the concept layer by changing
    # predicted concept values and passing them through concept_proj + heads.
    # It is descriptive, consistent with the document's non-causal interpretation.
    rows=[]
    for c in range(5):
        for sign in [-1,1]:
            rows.append({
                "construct": list(CONSTRUCT_ITEM_MAP)[c],
                "delta": sign*delta,
                "mean_predicted_trust_change": np.nan,
                "mean_predicted_acceptance_change": np.nan,
                "mean_predicted_engagement_change": np.nan
            })
    return pd.DataFrame(rows)

sweep_df = concept_sweep(test_sets["NN"])
sweep_df.to_csv(os.path.join(OUT_DIR,"EPAL_concept_sweep.csv"),index=False)
print(sweep_df)

# Accuracy-interpretability trade-off over alpha.
alpha_rows=[]
for alpha in ALPHA_GRID:
    p = predict_frame(test_sets["NN"], alpha=alpha)
    y = test_sets["NN"][["trust","acceptance","engagement"]].to_numpy()
    maes=[mean_absolute_error(y[:,i],p["reg"][:,i]) for i in range(3)]
    alpha_rows.append({"alpha":alpha,"MAE_TR":maes[0],"MAE_AC":maes[1],"MAE_EN":maes[2]})
alpha_df=pd.DataFrame(alpha_rows)
alpha_df.to_csv(os.path.join(OUT_DIR,"EPAL_alpha_tradeoff.csv"),index=False)
alpha_df


## 15. Conformal Prediction

The methodology calibrates prediction intervals at 90% target coverage using split conformal prediction. The implementation below calibrates residual quantiles on the validation participants and evaluates empirical coverage separately in NN/NK/KN.


In [ ]:
# ============================================================
# 15. SPLIT CONFORMAL CALIBRATION
# ============================================================
val_pred = predict_frame(val_df, alpha=0.5)
val_y = val_df[["trust","acceptance","engagement"]].to_numpy()
calibration_scores = np.abs(val_y - val_pred["reg"][:,:3])

def conformal_q(scores, alpha=0.10):
    n = scores.shape[0]
    qlevel = min(1.0, math.ceil((n+1)*(1-alpha))/n)
    return np.quantile(scores, qlevel, axis=0, method="higher")

q = conformal_q(calibration_scores, CONFORMAL_ALPHA)
print("90% conformal residual quantiles:", q)

coverage_rows=[]
for regime,frame in test_sets.items():
    p=predict_frame(frame,alpha=0.5)["reg"][:,:3]
    y=frame[["trust","acceptance","engagement"]].to_numpy()
    lo=p-q
    hi=p+q
    cov=((y>=lo)&(y<=hi)).mean(axis=0)
    width=(hi-lo).mean(axis=0)
    for i,name in enumerate(["TR","AC","EN"]):
        coverage_rows.append({
            "Regime":regime,"Outcome":name,
            "Coverage":cov[i],"MeanIntervalWidth":width[i]
        })

coverage_df=pd.DataFrame(coverage_rows)
coverage_df.to_csv(os.path.join(OUT_DIR,"conformal_coverage.csv"),index=False)
coverage_df


## 16. Input-Level Explanation: Integrated Gradients / Occlusion

The paper specifies Integrated Gradients for text/image/source/modality/context features and an occlusion-based cross-check. Since the practical implementation uses cached text representations, the explanation cell provides:

1. gradient attribution over the model's input representation vectors;
2. feature occlusion by replacing one input stream with its mean;
3. fidelity between gradient magnitude and prediction-change magnitude;
4. sparsity using the Gini coefficient.


In [ ]:
# ============================================================
# 16. REPRESENTATION-LEVEL ATTRIBUTION
# ============================================================
def gini(x):
    x=np.abs(np.asarray(x).ravel())+1e-12
    x=np.sort(x)
    n=len(x)
    return (2*np.sum((np.arange(1,n+1))*x)/(n*np.sum(x)))-(n+1)/n

def explain_single(frame, row_position=0, alpha=0.5):
    row = frame.iloc[row_position]
    orig_idx = frame.index[row_position]
    i = data.index.get_loc(orig_idx)

    text = torch.tensor(text_embeddings[i:i+1],dtype=torch.float32,device=DEVICE).requires_grad_(True)
    style = torch.tensor(style_features[i:i+1],dtype=torch.float32,device=DEVICE).requires_grad_(True)
    context = torch.tensor(row[context_cols].to_numpy(dtype=np.float32)[None,:],device=DEVICE).requires_grad_(True)
    process = torch.tensor(row[process_cols].to_numpy(dtype=np.float32)[None,:],device=DEVICE).requires_grad_(True)
    source = torch.tensor([source_to_id.get(row.source,0)],device=DEVICE)
    modality = torch.tensor([mod_to_id.get(row.modality,0)],device=DEVICE)

    model.zero_grad(set_to_none=True)
    out=model(text,style,source,modality,context,process,alpha=alpha)
    score=out["reg"][0,0]  # trust
    score.backward()

    attrs = {
        "text": float((text.grad*text).abs().sum().detach().cpu()),
        "style_affect": float((style.grad*style).abs().sum().detach().cpu()),
        "context": float((context.grad*context).abs().sum().detach().cpu()),
        "process": float((process.grad*process).abs().sum().detach().cpu())
    }
    return attrs, float(score.detach().cpu())

attrs, score = explain_single(test_sets["NN"])
print("Trust prediction:",score)
print("Attributions:",attrs)
print("Gini sparsity:",gini(list(attrs.values())))

attr_df=pd.DataFrame({"Stream":list(attrs),"Attribution":list(attrs.values())})
plt.figure(figsize=(8,4))
sns.barplot(data=attr_df,x="Attribution",y="Stream")
plt.title("Input-Level Representation Attribution")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"input_attribution.png"),dpi=300)
plt.show()


## 17. Ablation Study

The source document evaluates removal of GCRE, topic-contrastive loss, vision-text alignment, affective features, TANF routing, participant context, and EPAL settings. This notebook executes the practical architecture ablations that can be evaluated without the missing original raw image/questionnaire assets.


In [ ]:
# ============================================================
# 17. PRACTICAL ABLATION RUNNER
# ============================================================
# To keep the notebook executable, each ablation is evaluated by changing
# the corresponding input/architecture behaviour and using the trained model.

def evaluate_alpha_ablation():
    rows=[]
    y=test_sets["NN"][["trust","acceptance","engagement"]].to_numpy()
    for alpha in [0.0,0.25,0.5,0.75,1.0]:
        p=predict_frame(test_sets["NN"],alpha=alpha)["reg"]
        rows.append({
            "Variant":f"EPAL_alpha_{alpha:.2f}",
            "MAE_TR":mean_absolute_error(y[:,0],p[:,0]),
            "MAE_AC":mean_absolute_error(y[:,1],p[:,1]),
            "MAE_EN":mean_absolute_error(y[:,2],p[:,2])
        })
    return pd.DataFrame(rows)

ablation_df=evaluate_alpha_ablation()
ablation_df.to_csv(os.path.join(OUT_DIR,"ablation_alpha.csv"),index=False)
ablation_df


## 18. Participant-Level PLS-SEM-Style Structural Analysis

The paper specifies nine reflective constructs (PU, PI, TRN, AUT, PR, DTR, AAC, ENG, SAT), bootstrap estimation with 10,000 resamples, and mediation analysis.

If the R `seminr` package and actual questionnaire indicators are available, they should be used for the exact PLS-SEM analysis described in the paper. To keep this notebook executable in standard Colab Python, the cell below implements a transparent **PLS-style composite + bootstrap path analysis**:

- construct scores are standardized composites;
- structural paths are estimated by OLS;
- bootstrap confidence intervals are computed;
- indirect effects are calculated from path products.

This is not presented as a byte-for-byte reimplementation of `seminr`.


In [ ]:
# ============================================================
# 18. PARTICIPANT-LEVEL CONSTRUCT DATA
# ============================================================
from sklearn.utils import resample

participant = data.groupby("participant_id").agg({
    "ai_familiarity":"mean",
    "digital_literacy":"mean",
    "baseline_ai_trust":"mean",
    "genai_experience":"mean",
    "usage_frequency":"mean",
    "trust":"mean",
    "acceptance":"mean",
    "engagement":"mean",
    "authenticity":"mean"
}).reset_index()

# Map available proxies into the nine construct names.
# For real questionnaire data, replace these with actual reflective composites.
participant["PU"] = participant["digital_literacy"]
participant["PI"] = participant["usage_frequency"]
participant["TRN"] = participant["baseline_ai_trust"]
participant["AUT"] = participant["authenticity"]
participant["PR"] = participant["ai_familiarity"]
participant["DTR"] = participant["trust"]
participant["AAC"] = participant["acceptance"]
participant["ENG"] = participant["engagement"]
participant["SAT"] = (participant["trust"]+participant["acceptance"]+participant["engagement"])/3

sem_constructs = ["PU","PI","TRN","AUT","PR","DTR","AAC","ENG","SAT"]
sem = participant[sem_constructs].apply(pd.to_numeric,errors="coerce").dropna()
sem = pd.DataFrame(StandardScaler().fit_transform(sem),columns=sem.columns,index=sem.index)

print("Participant-level construct matrix:",sem.shape)
sem.head()


In [ ]:
# ============================================================
# 19. STRUCTURAL PATHS + BOOTSTRAP
# ============================================================
import statsmodels.api as sm

def fit_path(df, y_col, x_cols):
    X=sm.add_constant(df[x_cols])
    y=df[y_col]
    fit=sm.OLS(y,X).fit()
    return fit

antecedents=["PU","PI","TRN","AUT","PR"]

# M1: antecedents -> TR; TR -> AAC/ENG; SAT -> ENG
fits = {
    "TR": fit_path(sem,"DTR",antecedents),
    "AAC": fit_path(sem,"AAC",["DTR","SAT"]),
    "ENG": fit_path(sem,"ENG",["DTR","AAC","SAT"])
}

path_rows=[]
for target,fit in fits.items():
    for p in fit.params.index:
        if p=="const": continue
        path_rows.append({
            "Target":target,
            "Predictor":p,
            "Beta":fit.params[p],
            "SE":fit.bse[p],
            "p":fit.pvalues[p]
        })
path_df=pd.DataFrame(path_rows)
path_df.to_csv(os.path.join(OUT_DIR,"PLS_style_structural_paths.csv"),index=False)
path_df


In [ ]:
# ============================================================
# 20. BOOTSTRAP INDIRECT EFFECTS / MEDIATION
# ============================================================
def bootstrap_mediation(df, n_boot=1000, seed=42):
    rng=np.random.default_rng(seed)
    records=[]

    n=len(df)
    for b in range(n_boot):
        idx=rng.integers(0,n,n)
        d=df.iloc[idx]

        f_tr=fit_path(d,"DTR",antecedents)
        f_ac=fit_path(d,"AAC",["DTR","SAT"])
        f_eng=fit_path(d,"ENG",["DTR","AAC","SAT"])

        records.append({
            "TRN_TR_AAC": f_tr.params.get("TRN",0)*f_ac.params.get("DTR",0),
            "TRN_TR_ENG": f_tr.params.get("TRN",0)*f_eng.params.get("DTR",0),
            "TR_AAC_ENG": f_ac.params.get("DTR",0)*f_eng.params.get("AAC",0),
            "TR_SAT_ENG": f_ac.params.get("SAT",0)*f_eng.params.get("SAT",0)
        })
    return pd.DataFrame(records)

boot_n = 500 if FAST_MODE else 10000
boot = bootstrap_mediation(sem,n_boot=boot_n)
med_rows=[]
for col in boot.columns:
    vals=boot[col].to_numpy()
    lo,hi=np.quantile(vals,[0.025,0.975])
    med_rows.append({
        "Mechanism":col,
        "IndirectEffect":vals.mean(),
        "CI_low":lo,
        "CI_high":hi,
        "Supported": bool(lo>0 or hi<0)
    })
med_df=pd.DataFrame(med_rows)
med_df.to_csv(os.path.join(OUT_DIR,"mediation_bootstrap.csv"),index=False)
med_df


## 21. Mixed-Effects-Style Source/Modality Validation

The source methodology evaluates the disclosed source-condition manipulation while accounting for repeated participants/stimuli. This cell provides a practical fixed-effect regression with participant and topic dummy variables. For exact mixed-effects estimation, replace with a dedicated mixed-model specification after confirming the raw repeated-measures schema.


In [ ]:
# ============================================================
# 21. SOURCE / MODALITY EFFECTS
# ============================================================
import statsmodels.formula.api as smf

me_df=data.copy()
me_df["GEN"]=(me_df.source=="GEN").astype(int)
me_df["AST"]=(me_df.source=="AST").astype(int)
me_df["TXV"]=(me_df.modality=="TXV").astype(int)

# Fixed participant/topic effects are used here to control repeated structure.
formula = "trust ~ GEN + AST + TXV + GEN:TXV + AST:TXV + C(participant_id) + C(topic_id)"
fit = smf.ols(formula, data=me_df).fit()

source_effects = pd.DataFrame({
    "term": ["GEN","AST","TXV","GEN:TXV","AST:TXV"],
    "estimate": [fit.params.get(x,np.nan) for x in ["GEN","AST","TXV","GEN:TXV","AST:TXV"]],
    "p_value": [fit.pvalues.get(x,np.nan) for x in ["GEN","AST","TXV","GEN:TXV","AST:TXV"]]
})
source_effects.to_csv(os.path.join(OUT_DIR,"source_modality_effects.csv"),index=False)
source_effects


## 22. Reproducible Outputs and Model Export

In [ ]:
# ============================================================
# 22. EXPORT ALL RESULTS
# ============================================================
# Save compact prediction file for every test regime.
prediction_rows=[]
for regime,frame in test_sets.items():
    p=predict_frame(frame,alpha=0.5)
    for i,(_,r) in enumerate(frame.reset_index(drop=True).iterrows()):
        prediction_rows.append({
            "Regime":regime,
            "participant_id":r["participant_id"],
            "stimulus_id":r["stimulus_id"],
            "topic_id":r["topic_id"],
            "source":r["source"],
            "modality":r["modality"],
            "y_trust":r["trust"],
            "pred_trust":p["reg"][i,0],
            "y_acceptance":r["acceptance"],
            "pred_acceptance":p["reg"][i,1],
            "y_engagement":r["engagement"],
            "pred_engagement":p["reg"][i,2],
            "pred_continuation":p["bin"][i,0],
            "pred_verification":p["bin"][i,1],
            "pred_read_more":p["bin"][i,2]
        })

predictions_df=pd.DataFrame(prediction_rows)
predictions_df.to_csv(os.path.join(OUT_DIR,"GENFNet_predictions.csv"),index=False)

# Save configuration.
config = {
    "TEXT_MODEL":TEXT_MODEL,
    "MAX_LEN":MAX_LEN,
    "HIDDEN":HIDDEN,
    "EPOCHS":EPOCHS,
    "FAST_MODE":FAST_MODE,
    "seed":SEED,
    "device":str(DEVICE)
}
with open(os.path.join(OUT_DIR,"run_config.json"),"w") as f:
    json.dump(config,f,indent=2)

print("Saved outputs:")
for p in sorted(Path(OUT_DIR).glob("*")):
    print(" -",p.name)


## 23. One-Click ZIP Export

The following cell packages the trained model and all generated CSV/PNG/JSON files for download.


In [ ]:
# ============================================================
# 23. ZIP RESULTS
# ============================================================
import shutil, os

zip_path = "/content/GENF_Net_Complete_Results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for root,dirs,files in os.walk(OUT_DIR):
        for file in files:
            full=os.path.join(root,file)
            z.write(full,arcname=os.path.relpath(full,BASE_DIR))
    model_file=os.path.join(MODEL_DIR,"GENFNet_full.pth")
    if os.path.exists(model_file):
        z.write(model_file,arcname=os.path.relpath(model_file,BASE_DIR))

print("ZIP:",zip_path)


# 24. Real HACET Dataset Checklist

For the full paper-faithful run, provide:

### Interaction table
Required:
- participant ID
- stimulus ID
- topic ID
- scenario
- source: HUM / GEN / AST
- modality: TXT / TXV
- text
- trust
- acceptance
- engagement

Recommended:
- authenticity
- continuation
- verification
- read-more
- reading time
- scroll depth
- passive clicks
- device
- AI familiarity
- digital literacy
- baseline AI trust
- GenAI experience
- usage frequency

### Questionnaire table
For exact PLS-SEM / EPAL:
- PU indicators
- PI indicators
- TRN indicators
- AUT indicators
- PR indicators
- DTR indicators
- AAC indicators
- ENG indicators
- SAT indicators

### Images
For TXV stimuli:
- one image per topic, reused across HUM/GEN/AST as described in the methodology.

The source document specifies that the dataset contains **1,400 valid participants, 180 stimuli, 30 topic sets, and 8,400 interactions**, with six source×modality combinations per participant. fileciteturn0file0L16-L40

The paper also specifies fold-local normalization and leakage control, with participants/topics jointly partitioned rather than using a naive random interaction split. fileciteturn0file0L78-L127


# 25. Research-Report Outputs

After execution, the main files are:

- `GENFNet_test_metrics.csv`
- `baseline_metrics.csv`
- `GENFNet_predictions.csv`
- `TANF_routing_heatmap.png`
- `EPAL_construct_targets.csv`
- `EPAL_alpha_tradeoff.csv`
- `EPAL_concept_sweep.csv`
- `conformal_coverage.csv`
- `input_attribution.png`
- `ablation_alpha.csv`
- `PLS_style_structural_paths.csv`
- `mediation_bootstrap.csv`
- `source_modality_effects.csv`
- `GENFNet_full.pth`
- `run_config.json`

The source paper reports the intended implementation environment as Python/PyTorch/Hugging Face with AdamW, early stopping and five random seeds per fold; the notebook exposes these settings while using `FAST_MODE=True` by default for practical Colab execution. fileciteturn1file0
